[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/03-structured-outputs-and-tool-calling/code/structured_op_tool_calling.ipynb)

# Class 4.3: Structured outputs and tool calling

Make the model's output machine-usable. We define a Pydantic contract, validate good and bad responses, add a validate-and-retry loop, and run a full tool-calling round trip. This whole notebook runs offline: Pydantic and the tool registry are plain Python, and a small mock model stands in for the LLM so every cell executes anywhere. Swap the mock for the class 4.1 `chat` wrapper to use a real model.

## Setup

New library for this class:

```
pip install pydantic
```

## 1. The contract: a Pydantic model

In [1]:
# A Pydantic BaseModel is a typed "contract" for data. Each field has a type, and
# Pydantic enforces it. This is how we make an LLM's free text into a reliable object.
from pydantic import BaseModel, ValidationError
from typing import Literal

class Order(BaseModel):
    product: str                                   # must be text
    quantity: int                                  # must be a whole number
    priority: Literal["low", "normal", "high"]     # must be exactly one of these

# The model can emit its own JSON schema, which is what you hand an LLM so it knows
# the exact shape to return.
print(Order.model_json_schema()["required"])   # ['product', 'quantity', 'priority']

['product', 'quantity', 'priority']


## 2. A good response validates to a typed object

In [2]:
# model_validate_json parses a JSON string AND checks it against the contract in
# one step. If it passes, you get a typed Order object back.
raw = '{"product": "seats", "quantity": 3, "priority": "high"}'
order = Order.model_validate_json(raw)
print(order.product, "|", order.quantity, "|", order.priority)
# quantity came in as JSON and is now a real Python int, not a string:
print("quantity is an int:", isinstance(order.quantity, int))
# -> seats | 3 | high   then   quantity is an int: True

seats | 3 | high
quantity is an int: True


## 3. A malformed response raises a clear error

In [3]:
# When the data violates the contract, Pydantic raises ValidationError instead of
# silently passing bad data downstream. Here "three" is not a valid integer.
bad = '{"product": "seats", "quantity": "three", "priority": "high"}'
try:
    Order.model_validate_json(bad)
except ValidationError as e:
    # e.errors() is a list of dicts describing each problem: where it was and why.
    print(e.errors()[0]["loc"], "->", e.errors()[0]["msg"])
# -> ('quantity',) -> Input should be a valid integer, unable to parse string as an integer

('quantity',) -> Input should be a valid integer, unable to parse string as an integer


## 4. Validate and retry, under a cap

In [ ]:
import logging
# Show INFO logs (llm.py logs every model call; we add our own below).
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
import re
import llm   # real provider wrapper (gemini/groq/ollama via .env)

# Models sometimes wrap JSON in ``` fences or add a sentence around it. Grab the
# first {...} block so json parsing is robust.
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text

# The real reliability pattern: ask a REAL model for JSON, validate it against the
# Pydantic contract, and if it fails, feed the exact error back and try again, up
# to a cap. This is a genuine round trip; set up a provider before running.
def extract_order(note, max_tries=3):
    messages = [{"role": "user", "content":
        "Extract the order as JSON with keys product (string), quantity (integer), "
        "and priority (one of low, normal, high). Return ONLY the JSON.\n"
        f"Note: {note}"}]
    for attempt in range(1, max_tries + 1):
        raw = llm.chat(messages)                              # REAL model call
        try:
            return Order.model_validate_json(extract_json(raw)), attempt   # success
        except ValidationError as e:
            logging.warning(f"got valid order on attempt {attempt}: {raw}")
            # Append the specific error so the next attempt can correct itself.
            messages.append({"role": "user",
                             "content": f"That failed validation: {e.errors()[0]['msg']}. Return ONLY valid JSON."})
        logging.error(f"retrying after {attempt} failed attempts")
    raise RuntimeError(f"no valid order after {max_tries} tries")

order, tries = extract_order("3 seats, high priority")
print(f"got {order.quantity} {order.product} at {order.priority} priority in {tries} tries")
# A capable model usually succeeds on the first try; a weaker one may need the
# retry. Either way the output is a validated Order, never raw unchecked text.

INFO | google_genai.models | AFC is enabled with max remote calls: 10.
WARNING | google_genai.models | Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=2.482s


got 3 seats at high priority in 1 tries


## 5. Tool calling: define a tool and its schema

In [5]:
# Tool calling lets the model trigger YOUR code. A tool is just a normal Python
# function, plus a JSON schema that tells the model its name, what it does, and
# what arguments it takes, so the model knows when and how to ask for it.
INVOICES = {"INV-42": "paid", "INV-77": "overdue"}

def get_invoice_status(invoice_id: str) -> str:
    return INVOICES.get(invoice_id, "unknown")        # look up status, default "unknown"

TOOLS = {
    "get_invoice_status": {
        "fn": get_invoice_status,                     # the actual function to run
        "schema": {                                  # the description the model sees
            "name": "get_invoice_status",
            "description": "Look up the current status of an invoice by its id.",
            "parameters": {"type": "object",
                           "properties": {"invoice_id": {"type": "string"}},
                           "required": ["invoice_id"]},
        },
    }
}
print(TOOLS["get_invoice_status"]["schema"]["description"])

Look up the current status of an invoice by its id.


## 6. The three-step round trip

In [6]:
import json, re, logging
import llm
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text

# We ask the model to reply with ONE JSON object: either a tool call or a final
# answer. This is "manual" tool calling and works with any provider.
SYSTEM = (
    "You can use one tool, get_invoice_status(invoice_id). Reply with ONE JSON "
    "object and nothing else.\n"
    'To call the tool: {"tool": "get_invoice_status", "arguments": {"invoice_id": "INV-42"}}\n'
    'When you have the answer:  {"final": "Invoice INV-42 is paid."}'
)

def run(question, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for _ in range(max_steps):                       # step cap: never loop forever
        raw = llm.chat(messages)                     # REAL model call
        try:
            step = json.loads(extract_json(raw))
        except json.JSONDecodeError:
            messages.append({"role": "user", "content": "Reply with ONLY one JSON object."})
            continue
        if "final" in step:                          # model is done
            return step["final"]
        fn = TOOLS[step["tool"]]["fn"]               # 2. your code runs the tool
        result = fn(**step["arguments"])             # **unpacks {"invoice_id": ...}
        messages.append({"role": "assistant", "content": json.dumps(step)})     # 1. the request
        messages.append({"role": "user", "content": f"Tool result: {result}"})  # 3. result back
    return "stopped: step cap reached"

print(run("What is the status of INV-42?"))
print(run("Is INV-77 settled?"))
# Expected: the model requests get_invoice_status, your code runs it, and the model
# uses the result to answer, e.g. "Invoice INV-42 is paid." / "... is overdue."

INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=3.927s
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=3.969s


Invoice INV-42 is paid.


INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=2.844s
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=1.647s


Invoice INV-77 is not settled; its status is overdue.


In [7]:
print(run("Is INV-37 settled?"))

INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=4.554s
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | course.llm | llm call provider=gemini model=gemini-3.5-flash-lite latency=4.892s


I'm sorry, but I couldn't find an invoice with the ID INV-37.


## Recap

A Pydantic model is the contract: good JSON becomes a typed object, bad JSON fails loudly where you can see it, and a capped retry loop recovers from the occasional malformed response. Tool calling is the model requesting one of your functions with typed arguments; your code runs it and feeds the result back. That round trip in a loop is an agent, which is Module 5. Next class: embeddings and vector databases.

**Companion file in this folder:** `pydantic_basics.ipynb` is a focused tour of Pydantic (constraints, defaults, nested models, JSON in and out), the library you will reuse for tool schemas in Module 5 and FastAPI in Module 6.